In [11]:
import requests
from bs4 import BeautifulSoup
import pdfplumber
import pandas as pd
import re
from io import BytesIO

BASE = "https://airerm.mma.gob.cl/calidad-del-aire/consolidados-de-calidad-del-aire-ano-{}"
YEARS = range(2018, 2019)

STATES = ["Bueno", "Regular", "Alerta", "Preemergencia", "Emergencia"]
STATE_REGEX = re.compile(r"\b(" + "|".join(STATES) + r")\b", re.IGNORECASE)


def extract_first_state_from_pdf(url):
    try:
        r = requests.get(url, timeout=40)
        r.raise_for_status()
    except Exception:
        return None

    try:
        with pdfplumber.open(BytesIO(r.content)) as pdf:
            parts = []
            for page in pdf.pages:
                txt = page.extract_text()
                if txt:
                    parts.append(txt)

        if not parts:
            return None

        text = "\n".join(parts)

    except Exception:
        return None

    # normalize spacing and case
    text_norm = re.sub(r"\s+", " ", text).upper()

    # anchors used in GEC PDFs
    start_anchor = "CONDICIÓN DE CALIDAD DEL AIRE"
    end_anchor   = "CONDICIÓN METEOROLÓGICA"

    start = text_norm.find(start_anchor)

    if start != -1:
        sub = text_norm[start:]

        end = sub.find(end_anchor)
        if end != -1:
            sub = sub[:end]

        # search only inside the air quality block
        m = re.search(
            r"\b(BUENO|REGULAR|ALERTA|PREEMERGENCIA|EMERGENCIA)\b",
            sub,
            re.IGNORECASE
        )

        if m:
            found = m.group(1).lower()
            for s in STATES:
                if s.lower() == found:
                    return s

    # -------------------------------
    # Fallback (if anchors not found)
    # -------------------------------
    m = STATE_REGEX.search(text)
    if not m:
        return None

    found = m.group(1).lower()
    for s in STATES:
        if s.lower() == found:
            return s

    return None


rows = []

for year in YEARS:
    page_url = BASE.format(year)
    print(f"\nScraping {page_url}")

    try:
        r = requests.get(page_url, timeout=40)
        r.raise_for_status()
    except Exception as e:
        print("  -> page not available")
        continue

    soup = BeautifulSoup(r.text, "html.parser")

    for a in soup.select("a[href]"):
        href = a["href"].strip()
        raw_txt = a.get_text(" ", strip=True)

        m = re.search(r"(\d{2})\s*[-–]\s*(\d{2})\s*[-–]\s*(\d{4})", raw_txt)

        if not m:
            continue

        date_txt = f"{m.group(1)}-{m.group(2)}-{m.group(3)}"

        # more robust pdf detection
        if ".pdf" not in href.lower():
            continue

        print(f"  -> {date_txt}")

        state = extract_first_state_from_pdf(href)

        rows.append(
            {
                "date": date_txt,
                "state": state
            }
        )


# -----------------------
# SAFE DataFrame creation
# -----------------------
if len(rows) == 0:
    print("\nNo rows were collected. The site structure may have changed.")
    df = pd.DataFrame(columns=["date", "state"])
else:
    df = pd.DataFrame(rows)
    df = df.sort_values("date")

df.to_csv(
    r"C:\Users\black\Downloads\gec_states_2018_2024.csv",
    index=False,
    encoding="utf-8"
)

print("\nSaved: gec_states_2018_2024.csv")



Scraping https://airerm.mma.gob.cl/calidad-del-aire/consolidados-de-calidad-del-aire-ano-2018
  -> 31-08-2018
  -> 30-08-2018
  -> 29-08-2018
  -> 28-08-2018
  -> 27-08-2018
  -> 26-08-2018
  -> 25-08-2018
  -> 24-08-2018
  -> 23-08-2018
  -> 22-08-2018
  -> 21-08-2018
  -> 20-08-2018
  -> 19-08-2018
  -> 18-08-2018
  -> 17-08-2018
  -> 16-08-2018
  -> 15-08-2018
  -> 14-08-2018
  -> 13-08-2018
  -> 12-08-2018
  -> 11-08-2018
  -> 10-08-2018
  -> 09-08-2018
  -> 08-08-2018
  -> 07-08-2018
  -> 06-08-2018
  -> 05-08-2018
  -> 04-08-2018
  -> 03-08-2018
  -> 02-08-2018
  -> 01-08-2018
  -> 31-07-2018
  -> 30-07-2018
  -> 29-07-2018
  -> 28-07-2018
  -> 27-07-2018
  -> 26-07-2018
  -> 25-07-2018
  -> 24-07-2018
  -> 23-07-2018
  -> 22-07-2018
  -> 21-07-2018
  -> 20-07-2018
  -> 19-07-2018
  -> 18-07-2018
  -> 17-07-2018
  -> 16-07-2018
  -> 15-07-2018
  -> 14-07-2018
  -> 13-07-2018
  -> 12-07-2018
  -> 11-07-2018
  -> 10-07-2018
  -> 09-07-2018
  -> 08-07-2018
  -> 07-07-2018
  -> 06-0

In [13]:
import requests
from bs4 import BeautifulSoup
import pdfplumber
import pandas as pd
import re
from io import BytesIO
import unicodedata

BASE = "https://airerm.mma.gob.cl/calidad-del-aire/consolidados-de-calidad-del-aire-ano-{}"
YEARS = range(2018, 2019)

STATES = ["Bueno", "Regular", "Alerta", "Preemergencia", "Emergencia"]
STATE_REGEX = re.compile(r"\b(" + "|".join(STATES) + r")\b", re.IGNORECASE)


def strip_accents(s):
    return "".join(
        c for c in unicodedata.normalize("NFD", s)
        if unicodedata.category(c) != "Mn"
    )


def extract_first_state_from_pdf(url):
    try:
        r = requests.get(url, timeout=40)
        r.raise_for_status()
    except Exception:
        return None

    try:
        with pdfplumber.open(BytesIO(r.content)) as pdf:
            parts = []
            for page in pdf.pages:
                txt = page.extract_text()
                if txt:
                    parts.append(txt)

        if not parts:
            return None

        text = "\n".join(parts)

    except Exception:
        return None

    # -----------------------------------------
    # normalize: spaces, case, and accents
    # -----------------------------------------
    text_norm = re.sub(r"\s+", " ", text).upper()
    text_norm = strip_accents(text_norm)

    start_anchor = strip_accents("CONDICIÓN DE CALIDAD DEL AIRE").upper()
    end_anchor   = strip_accents("CONDICIÓN METEOROLÓGICA").upper()

    start = text_norm.find(start_anchor)

    if start != -1:
        sub = text_norm[start:]

        end = sub.find(end_anchor)
        if end != -1:
            sub = sub[:end]

        m = re.search(
            r"\b(BUENO|REGULAR|ALERTA|PREEMERGENCIA|EMERGENCIA)\b",
            sub
        )

        if m:
            found = m.group(1).lower()
            for s in STATES:
                if s.lower() == found:
                    return s

    # -------------------------------
    # fallback (whole document)
    # -------------------------------
    m = STATE_REGEX.search(text)
    if not m:
        return None

    found = m.group(1).lower()
    for s in STATES:
        if s.lower() == found:
            return s

    return None


rows = []

for year in YEARS:
    page_url = BASE.format(year)
    print(f"\nScraping {page_url}")

    try:
        r = requests.get(page_url, timeout=40)
        r.raise_for_status()
    except Exception:
        print("  -> page not available")
        continue

    soup = BeautifulSoup(r.text, "html.parser")

    for a in soup.select("a[href]"):
        href = a["href"].strip()
        raw_txt = a.get_text(" ", strip=True)

        m = re.search(r"(\d{2})\s*[-–]\s*(\d{2})\s*[-–]\s*(\d{4})", raw_txt)

        if not m:
            continue

        date_txt = f"{m.group(1)}-{m.group(2)}-{m.group(3)}"

        # only keep PDF links
        if ".pdf" not in href.lower():
            continue

        print(f"  -> {date_txt}")

        state = extract_first_state_from_pdf(href)

        rows.append(
            {
                "date": date_txt,
                "state": state
            }
        )


# -----------------------
# SAFE DataFrame creation
# -----------------------
if len(rows) == 0:
    print("\nNo rows were collected. The site structure may have changed.")
    df = pd.DataFrame(columns=["date", "state"])
else:
    df = pd.DataFrame(rows)
    df = df.sort_values("date")

df.to_csv(
    r"C:\Users\black\Downloads\gec_states_2018_2024.csv",
    index=False,
    encoding="utf-8"
)

print("\nSaved: gec_states_2018_2024.csv")



Scraping https://airerm.mma.gob.cl/calidad-del-aire/consolidados-de-calidad-del-aire-ano-2018
  -> 31-08-2018
  -> 30-08-2018
  -> 29-08-2018
  -> 28-08-2018
  -> 27-08-2018
  -> 26-08-2018
  -> 25-08-2018
  -> 24-08-2018
  -> 23-08-2018
  -> 22-08-2018
  -> 21-08-2018
  -> 20-08-2018
  -> 19-08-2018
  -> 18-08-2018
  -> 17-08-2018
  -> 16-08-2018
  -> 15-08-2018
  -> 14-08-2018
  -> 13-08-2018
  -> 12-08-2018
  -> 11-08-2018
  -> 10-08-2018
  -> 09-08-2018
  -> 08-08-2018
  -> 07-08-2018
  -> 06-08-2018
  -> 05-08-2018
  -> 04-08-2018
  -> 03-08-2018
  -> 02-08-2018
  -> 01-08-2018
  -> 31-07-2018
  -> 30-07-2018
  -> 29-07-2018
  -> 28-07-2018
  -> 27-07-2018
  -> 26-07-2018
  -> 25-07-2018
  -> 24-07-2018
  -> 23-07-2018
  -> 22-07-2018
  -> 21-07-2018
  -> 20-07-2018
  -> 19-07-2018
  -> 18-07-2018
  -> 17-07-2018
  -> 16-07-2018
  -> 15-07-2018
  -> 14-07-2018
  -> 13-07-2018
  -> 12-07-2018
  -> 11-07-2018
  -> 10-07-2018
  -> 09-07-2018
  -> 08-07-2018
  -> 07-07-2018
  -> 06-0

In [20]:
import requests
from bs4 import BeautifulSoup
import pdfplumber
import pandas as pd
import re
from io import BytesIO
import unicodedata

BASE = "https://airerm.mma.gob.cl/calidad-del-aire/consolidados-de-calidad-del-aire-ano-{}"
YEARS = range(2018, 2025)

STATES = ["Bueno", "Regular", "Alerta", "Preemergencia", "Emergencia"]


def strip_accents(s):
    return "".join(
        c for c in unicodedata.normalize("NFD", s)
        if unicodedata.category(c) != "Mn"
    )


def normalize_letters_only(s):
    s = strip_accents(s.upper())
    s = re.sub(r"[^A-Z]", "", s)
    return s


def extract_first_state_from_pdf(url):
    try:
        r = requests.get(url, timeout=40)
        r.raise_for_status()
    except Exception:
        return None

    try:
        with pdfplumber.open(BytesIO(r.content)) as pdf:
            parts = []
            for page in pdf.pages:
                txt = page.extract_text()
                if txt:
                    parts.append(txt)

        if not parts:
            return None

        text = "\n".join(parts)

    except Exception:
        return None

    # -----------------------------------------
    # aggressive normalization of whole document
    # -----------------------------------------
    letters = normalize_letters_only(text)

    # allow first letter to be missing
    damaged_patterns = {
        "Bueno": ["BUENO", "UENO"],
        "Regular": ["REGULAR", "EGULAR"],
        "Alerta": ["ALERTA", "LERTA"],
        "Preemergencia": ["PREEMERGENCIA", "REEMERGENCIA"],
        "Emergencia": ["EMERGENCIA", "MERGENCIA"]
    }

    earliest = None
    earliest_state = None

    for state, pats in damaged_patterns.items():
        for p in pats:
            idx = letters.find(p)
            if idx != -1:
                if earliest is None or idx < earliest:
                    earliest = idx
                    earliest_state = state

    return earliest_state


rows = []

for year in YEARS:
    page_url = BASE.format(year)
    print(f"\nScraping {page_url}")

    try:
        r = requests.get(page_url, timeout=40)
        r.raise_for_status()
    except Exception:
        print("  -> page not available")
        continue

    soup = BeautifulSoup(r.text, "html.parser")

    for a in soup.select("a[href]"):
        href = a["href"].strip()
        raw_txt = a.get_text(" ", strip=True)

        m = re.search(r"(\d{2})\s*[-–]\s*(\d{2})\s*[-–]\s*(\d{4})", raw_txt)

        if not m:
            continue

        date_txt = f"{m.group(1)}-{m.group(2)}-{m.group(3)}"

        # only keep PDF links
        if ".pdf" not in href.lower():
            continue

        print(f"  -> {date_txt}")

        state = extract_first_state_from_pdf(href)

        rows.append(
            {
                "date": date_txt,
                "state": state
            }
        )


# -----------------------
# SAFE DataFrame creation
# -----------------------
if len(rows) == 0:
    print("\nNo rows were collected. The site structure may have changed.")
    df = pd.DataFrame(columns=["date", "state"])
else:
    df = pd.DataFrame(rows)
    df = df.sort_values("date")

df.to_csv(
    r"C:\Users\black\Downloads\gec_states_2018_2024.csv",
    index=False,
    encoding="utf-8"
)

print("\nSaved: gec_states_2018_2024.csv")



Scraping https://airerm.mma.gob.cl/calidad-del-aire/consolidados-de-calidad-del-aire-ano-2018
  -> 31-08-2018
  -> 30-08-2018
  -> 29-08-2018
  -> 28-08-2018
  -> 27-08-2018
  -> 26-08-2018
  -> 25-08-2018
  -> 24-08-2018
  -> 23-08-2018
  -> 22-08-2018
  -> 21-08-2018
  -> 20-08-2018
  -> 19-08-2018
  -> 18-08-2018
  -> 17-08-2018
  -> 16-08-2018
  -> 15-08-2018
  -> 14-08-2018
  -> 13-08-2018
  -> 12-08-2018
  -> 11-08-2018
  -> 10-08-2018
  -> 09-08-2018
  -> 08-08-2018
  -> 07-08-2018
  -> 06-08-2018
  -> 05-08-2018
  -> 04-08-2018
  -> 03-08-2018
  -> 02-08-2018
  -> 01-08-2018
  -> 31-07-2018
  -> 30-07-2018
  -> 29-07-2018
  -> 28-07-2018
  -> 27-07-2018
  -> 26-07-2018
  -> 25-07-2018
  -> 24-07-2018
  -> 23-07-2018
  -> 22-07-2018
  -> 21-07-2018
  -> 20-07-2018
  -> 19-07-2018
  -> 18-07-2018
  -> 17-07-2018
  -> 16-07-2018
  -> 15-07-2018
  -> 14-07-2018
  -> 13-07-2018
  -> 12-07-2018
  -> 11-07-2018
  -> 10-07-2018
  -> 09-07-2018
  -> 08-07-2018
  -> 07-07-2018
  -> 06-0

In [19]:
df.to_csv(
    r"C:\Users\black\Downloads\gec_states_2018_2024.csv",
    index=False,
    encoding="utf-8"
)